# Stage 3 (last experimental stage): baselines, cross-domain validation, qualitative + failure analysis
GPU runtime. Rough budget on a T4: main dataset 10 runs x ~5 min, cross-domain 2 targets x 2 encoders x 3 seeds = 12 runs x ~4 min (robustness is skipped here: v2 already has it).

In [ ]:
!rm -rf semantic-chromatic-retrieval && git clone https://github.com/MokshdaSharma/semantic-chromatic-retrieval.git
%cd semantic-chromatic-retrieval
!pip install -q -r requirements.txt

In [ ]:
# Re-download WITH masterCategory + productDisplayName (needed for the cross-domain split). Same shuffled subsample as before.
!python scripts/prepare_fashion_small.py --out /content/fashion_small --n 30000
!head -3 /content/fashion_small/styles.csv

In [ ]:
import subprocess, sys
def run(args): print(' '.join(args)); return subprocess.run([sys.executable, '-m', 'cr.run'] + args).returncode
SEEDS = [0, 1, 2, 3, 4]
for s in SEEDS:
    for e in ['clip', 'dinov2']:
        a = ['--dataset', 'fashion_small', '--root', '/content/fashion_small', '--encoder', e, '--seed', str(s), '--no_robustness', '--out', f'results/v3/main_{e}_s{s}']
        if s == 0: a.append('--qualitative')   # example grids + failure analysis for seed 0
        run(a)

In [ ]:
XD_SEEDS = [0, 1, 2]
for target in ['Footwear', 'Accessories']:
    for s in XD_SEEDS:
        for e in ['clip', 'dinov2']:
            a = ['--dataset', 'fashion_xdomain', '--root', '/content/fashion_small', '--encoder', e, '--seed', str(s), '--source_master', 'Apparel', '--target_master', target, '--no_robustness', '--out', f'results/v3/xd_{target}_{e}_s{s}']
            if s == 0: a.append('--qualitative')
            run(a)

## Optional: a REAL second dataset (you choose it; I could not verify any specific repo)
Pick a HuggingFace image dataset with an image, a category and a colour label or a caption mentioning one colour; check its card (size, licence). Fill in the placeholders and run.

In [ ]:
# !python scripts/prepare_hf_labeled.py --repo <hf/repo> --split train --image_col <img> --category_col <cat> --colour_text_col <caption> --n 20000 --out /content/second
# for e in ['clip','dinov2']:
#     for s in [0,1,2]:
#         run(['--dataset','generic_labeled','--name','second','--root','/content/second','--encoder',e,'--seed',str(s),'--no_robustness','--out',f'results/v3/second_{e}_s{s}'])

In [ ]:
# Aggregate ONE dataset at a time
!python -m cr.aggregate --runs results/v3/main_* --out results/v3_tables/main
!python -m cr.aggregate --runs results/v3/xd_Footwear_* --out results/v3_tables/xd_Footwear
!python -m cr.aggregate --runs results/v3/xd_Accessories_* --out results/v3_tables/xd_Accessories
!python -m cr.paper_tables --runs results/v3/main_* --tables results/v3_tables/main --out results/v3_tables/main/paper
!python -m cr.compare_datasets --tables main=results/v3_tables/main xd_Footwear=results/v3_tables/xd_Footwear xd_Accessories=results/v3_tables/xd_Accessories --out results/v3_tables/cross_dataset

In [ ]:
# v3 must reproduce v2 for unchanged methods (needs results/v2 from the previous stage in the repo)
!python scripts/check_v2_v3_consistency.py results/v2 results/v3

In [ ]:
!zip -qr results_v3.zip results/v3 results/v3_tables -x '*/cache/*'
from google.colab import files; files.download('results_v3.zip')